# Step 7 — LangGraph Orchestration + Human-in-the-Loop

All six agents as a `StateGraph` with a **verifier retry loop** and a **human interrupt** before
finalizing (`docs/orchestration.md`). State is serializable so the run pauses at the interrupt and
resumes after the human approves/edits. Node updates stream — this is exactly what the
**CopilotKit** UI renders (`docs/ui.md`).

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot.graph import build_graph, initial_state
from langgraph.types import Command

graph = build_graph()
cfg = {"configurable": {"thread_id": "nb-demo"}}

print("=== streaming agent updates until the human gate ===")
for ev in graph.stream(initial_state(settings.ticker, settings.period), cfg, stream_mode="updates"):
    for node, upd in ev.items():
        if isinstance(upd, dict) and upd.get("messages"):
            print(f"[{node:9}] {upd['messages'][-1][1]}")

paused = graph.get_state(cfg)
print("\nPAUSED before:", paused.next, "(human-in-the-loop interrupt)")
assert paused.next == ("hitl",)

/Users/v843010/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/v843010/Library/Python/3.9/lib/python/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


=== streaming agent updates until the human gate ===
[extract  ] Extracted 15 grounded facts; 2 peers.


[wiki     ] Indexed wiki (memory).
[sentiment] Net sentiment -0.05; 0 concerns.
[compare  ] Peer leads=['gross_margin', 'operating_margin', 'roe', 'roic', 'roa', 'ttm_eps'] lags=[].
[predict  ] Predicted 6 hard questions.
[draft    ] Drafted script/deck/Q&A (attempt 1).
[verify   ] Verification passed.

PAUSED before: ('hitl',) (human-in-the-loop interrupt)


## Human reviews, then resumes (approve / edit)

In the app this is `useLangGraphInterrupt` (CopilotKit) or a Flutter `UserActionEvent`. Here we
resume with an approval.

In [3]:
for ev in graph.stream(Command(resume=True, update={"human_feedback": {"decision": "approve"}}),
                       cfg, stream_mode="updates"):
    for node, upd in ev.items():
        if isinstance(upd, dict) and upd.get("messages"):
            print(f"[{node:9}] {upd['messages'][-1][1]}")

final = graph.get_state(cfg)
print("\nfinal next:", final.next, "(empty = done) | verification passed:",
      final.values["verification"].passed)
assert final.next == () and final.values["verification"].passed
print("\nEND-TO-END PIPELINE OK: extract -> wiki -> sentiment -> compare -> predict -> draft ->"
      " verify -> [human] -> finalize")

[hitl     ] Human decision: approve.
[finalize ] Finalized artifacts: script, deck outline, Q&A cheat sheet.

final next: () (empty = done) | verification passed: True

END-TO-END PIPELINE OK: extract -> wiki -> sentiment -> compare -> predict -> draft -> verify -> [human] -> finalize


### Going live on the MI300X
Flip `.env`: `LLM_BACKEND=vllm`, `EMBEDDING_BACKEND=vllm`, `QDRANT_MODE=docker`,
`USE_MOCK_DATA=false`, `SENTIMENT_BACKEND=finbert`. No node code changes — the agents call the
vLLM OpenAI-compatible endpoints (`docs/rocm-vllm.md`).

**Next (Step 8):** the critical fine-tuning of the Predictive Analyst (Unsloth) — `docs/finetuning.md`.